In [1]:
!pip install -U unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
GPU: Tesla T4


In [3]:
model_name = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.15: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [5]:
print(type(model))
print(type(tokenizer))

<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [6]:
from google.colab import files

uploaded = files.upload()

Saving data.json to data.json


In [7]:
import json

with open("data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of examples:", len(data))
print(data[0])

Number of examples: 735
{'instruction': 'What is Python?', 'response': 'संदर्भ— Python is a high-level programming language.'}


In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 735
})


In [9]:
print(dataset[0])

{'instruction': 'What is Python?', 'response': 'संदर्भ— Python is a high-level programming language.'}


In [10]:
def format_chat(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [11]:
formatted_dataset = dataset.map(format_chat)

Map:   0%|          | 0/735 [00:00<?, ? examples/s]

In [12]:
print(formatted_dataset[0]["text"])

<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
<think>

</think>

संदर्भ— Python is a high-level programming language.<|im_end|>



In [13]:
print("Examples:", len(formatted_dataset))

for i in range(min(3, len(formatted_dataset))):
    print("\n--- Example", i + 1, "---")
    print(formatted_dataset[i]["text"])

Examples: 735

--- Example 1 ---
<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
<think>

</think>

संदर्भ— Python is a high-level programming language.<|im_end|>


--- Example 2 ---
<|im_start|>user
What is Django?<|im_end|>
<|im_start|>assistant
<think>

</think>

संदर्भ— Django is a Python web framework.<|im_end|>


--- Example 3 ---
<|im_start|>user
What is React?<|im_end|>
<|im_start|>assistant
<think>

</think>

संदर्भ— React is a JavaScript library for building user interfaces.<|im_end|>



In [14]:
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# model = FastLanguageModel.get_peft_model(
#     model,
#     r=16,
#     lora_alpha=16,
#     lora_dropout=0,
#     target_modules=[
#         "q_proj",
#         "k_proj",
#         "v_proj",
#         "o_proj",
#     ],
# )


Unsloth 2026.8.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [15]:
model.print_trainable_parameters()


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [17]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=1024,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        num_train_epochs=3,
        learning_rate=2e-4,

        logging_steps=1,

        # Important: don't create checkpoints during this experiment
        save_strategy="no",

        fp16=True,

        output_dir="outputs",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/735 [00:00<?, ? examples/s]

In [18]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

prompt = "What is a URL?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(
    tokenizer,
    skip_prompt=True
)

_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=100,
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=100) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Okay, the user is asking what a URL is. Let me start by recalling the basic definition. A URL is a reference to a resource, like a website, a file, or a service. It's used to locate information on the web. I should mention the components like the protocol, domain, path, and query parameters.

Wait, I need to make sure I explain it clearly without getting too technical. Maybe start with the purpose: to locate information. Then list the parts:


In [19]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 735 | Num Epochs = 3 | Total steps = 276
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.917642
2,4.120515
3,3.701745
4,3.022485
5,2.768442
6,2.191932
7,2.041682
8,1.829120
9,1.559092
10,1.320986


TrainOutput(global_step=276, training_loss=0.7222994411553162, metrics={'train_runtime': 461.766, 'train_samples_per_second': 4.775, 'train_steps_per_second': 0.598, 'total_flos': 263145062400000.0, 'train_loss': 0.7222994411553162, 'epoch': 3.0})

In [20]:
import torch
import transformers
import trl
import unsloth

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("Unsloth:", unsloth.__version__)

Torch: 2.11.0+cu128
Transformers: 5.5.0
TRL: 0.24.0
Unsloth: 2026.8.15


In [21]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

In [22]:
prompt = "What is the capital of Japan?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=100,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


user
What is the capital of Japan?
assistant
<think>

</think>

संदर्भ— The capital of Japan is Tokyo.


In [23]:
model.save_pretrained_gguf(
    "model",
    tokenizer,
    quantization_method=["f16", "q4_k_m"]  # export f16 too, for testing
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:19<00:00, 19.85s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:04<00:00,  4.94s/it]


Unsloth: Merge process complete. Saved to `/content/model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['f16', 'q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10360-mix-87da1a2 (app-b10360-mix-87da1a2-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf/qwen3-0.6b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generate

{'save_directory': 'model',
 'gguf_directory': 'model_gguf',
 'gguf_files': ['model_gguf/qwen3-0.6b.Q4_K_M.gguf',
  'model_gguf/qwen3-0.6b.F16.gguf'],
 'modelfile_location': 'model_gguf/Modelfile',
 'want_full_precision': True,
 'is_vlm': False,
 'fix_bos_token': False}

In [24]:
!ls -lh /content/model

total 1.2G
-rw-r--r-- 1 root root 4.8K Aug 12 08:13 chat_template.jinja
-rw-r--r-- 1 root root 1.6K Aug 12 08:13 config.json
-rw-r--r-- 1 root root  210 Aug 12 08:13 generation_config.json
-rw-r--r-- 1 root root 1.2G Aug 12 08:13 model.safetensors
-rw-r--r-- 1 root root  10K Aug 12 08:13 tokenizer_config.json
-rw-r--r-- 1 root root  11M Aug 12 08:13 tokenizer.json


In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")

The tokenizer you are loading from '/content/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model loaded successfully


In [26]:
import torch
import transformers
import unsloth

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Unsloth:", unsloth.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.5.0
Unsloth: 2026.8.15


In [27]:
print(model.config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "float16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_wind

In [28]:
attention = model.model.layers[0].self_attn

print(type(attention))
print("apply_qkv:", hasattr(attention, "apply_qkv"))
print("forward:", attention.forward)

<class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'>
apply_qkv: False
forward: <bound method Qwen3Attention_fast_forward of Qwen3Attention(
  (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
  (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
  (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
  (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
  (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
  (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
)>


In [29]:
print("attn_implementation:",
      getattr(model.config, "_attn_implementation", None))

attn_implementation: sdpa


In [30]:
print("model type:", type(model))
print("attention type:", type(model.model.layers[0].self_attn))

model type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
attention type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'>


In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/content/model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)

print(type(model.model.layers[0].self_attn))
print("apply_qkv:",
      hasattr(model.model.layers[0].self_attn, "apply_qkv"))

The tokenizer you are loading from '/content/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

<class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'>
apply_qkv: False


In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/content/model"

tokenizer = AutoTokenizer.from_pretrained(
    model_path
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)

print("Attention implementation:",
      getattr(model.config, "_attn_implementation", None))

print("Attention class:",
      type(model.model.layers[0].self_attn))

print("Forward:",
      model.model.layers[0].self_attn.forward)

print("apply_qkv:",
      hasattr(model.model.layers[0].self_attn, "apply_qkv"))

The tokenizer you are loading from '/content/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Attention implementation: eager
Attention class: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'>
Forward: <bound method Qwen3Attention_fast_forward of Qwen3Attention(
  (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
  (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
  (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
  (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
  (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
  (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
)>
apply_qkv: False


In [33]:
messages = [
    {
        "role": "user",
        "content": "What is the capital of China ?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(text)

<|im_start|>user
What is the capital of China ?<|im_end|>
<|im_start|>assistant



In [34]:
print(model.config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "float16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_wind

In [35]:
# model.save_pretrained("my_url_adapter")
# tokenizer.save_pretrained("my_url_adapter")

In [36]:
# model.save_pretrained_gguf(
#     "my_url_gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )

In [37]:
# import shutil

# shutil.make_archive(
#     "my_url_gguf_gguf",
#     "zip",
#     root_dir=".",
#     base_dir="my_url_gguf_gguf"
# )

In [38]:
# adapter_path = "./my_url_mergedd"

# model.save_pretrained(adapter_path)
# tokenizer.save_pretrained(adapter_path)

# print(f"Adapter saved to: {adapter_path}")

In [39]:
# import os

# for file in os.listdir(adapter_path):
#     print(file)

In [41]:
print(model.peft_config)

AttributeError: 'Qwen3ForCausalLM' object has no attribute 'peft_config'

In [1]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.5.0


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)

attention = model.model.layers[0].self_attn

print("Attention implementation:",
      model.config._attn_implementation)

print("Forward:", attention.forward)

print("apply_qkv:", hasattr(attention, "apply_qkv"))

The tokenizer you are loading from '/content/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Attention implementation: eager
Forward: <bound method Qwen3Attention.forward of Qwen3Attention(
  (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
  (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
  (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
  (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
  (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
  (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
)>
apply_qkv: False


In [3]:
messages = [
    {
        "role": "user",
        "content": "What is the capital of China?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

print(tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=100) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


user
What is the capital of China?
assistant
<think>

</think>

संदर्भ— The capital of China is Beijing.


In [ ]:
# model.save_pretrained_merged(
#     "./my_url_merged",
#     tokenizer,
#     save_method="merged_16bit",
# )

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
# import torch

# BASE_MODEL = model_name
# ADAPTER_PATH = "my_url_adapter"
# OUTPUT_PATH = "my_url_mergedd"

# print("Loading base model...")

# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )

# print("Base model loaded.")

# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# print("Loading adapter...")

# model = PeftModel.from_pretrained(
#     base_model,
#     ADAPTER_PATH,
# )

# print("Adapter loaded.")

# print("Merging adapter...")

# model = model.merge_and_unload()

# print("Adapter merged.")

# print("Saving merged model...")

# model.save_pretrained(
#     OUTPUT_PATH,
#     safe_serialization=True,
# )

# tokenizer.save_pretrained(OUTPUT_PATH)

# print("Merged model saved.")

In [ ]:
# from transformers import AutoTokenizer

# BASE_MODEL = model_name
# OUTPUT_PATH = "my_url_mergedd"

# tokenizer = AutoTokenizer.from_pretrained(
#     BASE_MODEL,
#     use_fast=False
# )

# tokenizer.save_pretrained(OUTPUT_PATH)

# print("Tokenizer saved.")

In [4]:
!ls -lh /content/model_gguf

total 1.5G
-rw-r--r-- 1 root root 1.7K Aug 12 08:15 Modelfile
-rw-r--r-- 1 root root 1.2G Aug 12 08:14 qwen3-0.6b.F16.gguf
-rw-r--r-- 1 root root 379M Aug 12 08:15 qwen3-0.6b.Q4_K_M.gguf


In [5]:
!cd /content && zip -r model.zip raw_model

  adding: model_gguf/ (stored 0%)
  adding: model_gguf/qwen3-0.6b.Q4_K_M.gguf (deflated 3%)
  adding: model_gguf/Modelfile (deflated 63%)
  adding: model_gguf/qwen3-0.6b.F16.gguf (deflated 23%)


In [6]:
!ls -lh /content/model_gguf.zip

-rw-r--r-- 1 root root 1.3G Aug 12 10:06 /content/model_gguf.zip


In [7]:
from google.colab import files

files.download("/content/model_gguf.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!find /content -maxdepth 3 -type f \( \
    -name "adapter_model.safetensors" -o \
    -name "adapter_model.bin" \
\) -print